In [ ]:
import os
import torch
import torchaudio
from torchaudio.functional import add_noise
from torchaudio.transforms import Spectrogram

In [ ]:
SR = 8000
N_FFT = 256             # 수정: 512 -> 256
N_BINS = N_FFT // 2 + 1
HOP_LENGTH = 128
DURATION_SEC = 2
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

In [ ]:
class SirenGRU(torch.nn.Module):
    def __init__(self):
        super().__init__()
        self.gru = torch.nn.GRU(
            input_size=N_BINS,
            hidden_size=128,
            num_layers=2,
            batch_first=True,
            bidirectional=False # 실시간 타겟이라 단방향 고정
        )
        self.fc = torch.nn.Linear(128, N_BINS)
        self.sigmoid = torch.nn.Sigmoid()

    def forward(self, x):
        out, _ = self.gru(x)
        return self.sigmoid(self.fc(out))

In [ ]:
''' 터미널에 
cd /workspaces/siren-detection
source .venv/bin/activate

sudo apt update
sudo apt install -y ffmpeg libavcodec-dev libavformat-dev libavutil-dev
sudo ldconfig

pip install --force-reinstall --no-cache-dir torchcodec
'''

if __name__ == "__main__":
    print(f"디바이스: {DEVICE}")
    
    model_path = "/workspaces/siren-detection/siren_gru.pth"

    model = SirenGRU().to(DEVICE)
    model.load_state_dict(torch.load(model_path, map_location=DEVICE))
    model.eval()

    stft = Spectrogram(
        n_fft=N_FFT, 
        hop_length=HOP_LENGTH, 
        power=1.0  # magnitude 출력 
    ).to(DEVICE)

    test_clean_path = "/workspaces/siren-detection/test/test_siren.wav"
    test_noise_path = "/workspaces/siren-detection/test/test_wind.wav"

    clean_wave, clean_sr = torchaudio.load(test_clean_path)
    if clean_wave.shape[0] > 1: # 스테레오면 모노로 변환
        clean_wave = torch.mean(clean_wave, dim=0, keepdim=True)
    if clean_sr != SR: # 샘플링 레이트가 다르면 리샘플링
        clean_wave = torchaudio.functional.resample(clean_wave, clean_sr, SR)
    
    noise_wave, noise_sr = torchaudio.load(test_noise_path)
    if noise_wave.shape[0] > 1: # 스테레오면 모노로 변환
        noise_wave = torch.mean(noise_wave, dim=0, keepdim=True)
    if noise_sr != SR: # 샘플링 레이트가 다르면 리샘플링
        noise_wave = torchaudio.functional.resample(noise_wave, noise_sr, SR)

    max_len = SR * DURATION_SEC
    if clean_wave.shape[1] > max_len:
        clean_wave = clean_wave[:, :max_len]
    else:
        clean_wave = torch.nn.functional.pad(clean_wave, (0, max_len - clean_wave.shape[1]))
        
    if noise_wave.shape[1] > max_len:
        noise_wave = noise_wave[:, :max_len]
    else:
        noise_wave = torch.nn.functional.pad(noise_wave, (0, max_len - noise_wave.shape[1]))

    # 사이렌 파워와 소음 파워가 1:1로 섞인 상황 테스트
    target_snr = torch.tensor([0.0]) # 0dB SNR : 사이렌과 소음이 동일한 파워로 섞임
    mixed_wave = add_noise(clean_wave, noise_wave, target_snr)
    
    torchaudio.save("/workspaces/siren-detection/test/test_input_mixed.wav", mixed_wave, SR)

    with torch.no_grad():
        # 혼합 신호의 STFT 스펙트로그램 추출
        m_gt = stft(mixed_wave.to(DEVICE)) # Shape: (1, 129, Time)

        X = m_gt[0].transpose(0, 1) 
        X = X.unsqueeze(0)          

        pred_mask = model(X) # 출력 차원: (1, Time, 129)

    # 최종 결과 행렬 복원
    # 배치 차원을 해제하고 다시 주파수 연산을 위해 (129, Time) 형태로 역전치
    mask_2d = pred_mask.squeeze(0).transpose(0, 1) 
    
    # 원본 STFT 에너지 텐서에 인공지능이 뱉은 필터 마스크를 직접 곱함
    enhanced_gt = m_gt[0] * mask_2d

    print(f"마스크 최소값 (소음 대역 차단력): {mask_2d.min().item():.4f}")
    print(f"마스크 최대값 (사이렌 대역 보존력): {mask_2d.max().item():.4f}")
    print("최소값이 0에 가깝고 최대값이 1에 가깝게 벌어지면 필터 학습이 성공.")

In [ ]:
with torch.no_grad():
    # 위상 확보를 위해 복소 STFT 재계산 (Spectrogram은 magnitude만 반환하므로)
    complex_spec = torch.stft(
        mixed_wave.to(DEVICE),
        n_fft=N_FFT,
        hop_length=HOP_LENGTH,
        window=torch.hann_window(N_FFT).to(DEVICE),
        return_complex=True
    )  # Shape: (1, 129, Time)

    phase = torch.angle(complex_spec[0])

    enhanced_complex = enhanced_gt * torch.exp(1j * phase)

    # ISTFT로 파형 복원
    enhanced_wave = torch.istft(
        enhanced_complex.unsqueeze(0),
        n_fft=N_FFT,
        hop_length=HOP_LENGTH,
        window=torch.hann_window(N_FFT).to(DEVICE),
        length=max_len
    )  # Shape: (1, max_len)

    torchaudio.save(
        "/workspaces/siren-detection/test/test_enhanced.wav",
        enhanced_wave.cpu(),
        SR
    )